# Step 5: Feature Engineering
Calculates business features: `total_order_value`, `delivery_days`, `delivery_delay`, `customer_order_count`, `average_order_value`, `seller_revenue`, and `repeat_customer_indicator`.


In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = os.getenv("MYSQL_PORT", "3306")
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "root")
MYSQL_DB = os.getenv("MYSQL_DB", "cart2insights_db")

mysql_uri = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
engine = create_engine(mysql_uri)
print(f"Connected to MySQL database: {MYSQL_DB} on {MYSQL_HOST}:{MYSQL_PORT}")

orders = pd.read_sql(text("SELECT * FROM orders"), engine)
order_items = pd.read_sql(text("SELECT * FROM order_items"), engine)

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_delay'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['is_delayed'] = orders['delivery_delay'] > 0

# Write engineered features to MySQL table orders_features
try:
    orders.to_sql('orders_features', con=engine, if_exists='replace', index=False)
    print("[SUCCESS] Saved orders_features table to MySQL.", flush=True)
except Exception as e:
    print(f"Note: {e}")

print("Engineered features preview:", flush=True)
print(orders[['order_id', 'delivery_days', 'delivery_delay', 'is_delayed']].head(), flush=True)


Connected to MySQL database: cart2insights_db on localhost:3306
[SUCCESS] Saved orders_features table to MySQL.
Engineered features preview:
                           order_id  delivery_days  delivery_delay  is_delayed
0  e481f51cbdc54678b7cc49136f2d6af7            8.0            -8.0       False
1  53cdb2fc8bc7dce0b6741e2150273451           13.0            -6.0       False
2  47770eb9100c2d0c44946d9cf07ec65d            9.0           -18.0       False
3  949d5b44dbf5de918fe9c16f97b45f8a           13.0           -13.0       False
4  ad21c59c0840e6cb83a9ceb5573f8159            2.0           -10.0       False
